In [3]:
# ---- INITIAL CHECKS ---- #
import importlib.util
import subprocess
import sys

# Packages and their import names (sometimes different from pip names)
packages = {
    "selenium": "selenium",
    "webdriver-manager": "webdriver_manager",
    "requests": "requests",
    "pandas": "pandas",
    "tabula-py": "tabula",
    "pdfplumber": "pdfplumber",
    "pyarrow": "pyarrow",
}

for pip_name, import_name in packages.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
    else:
        print(f"{pip_name} already installed.")

# ---- Actual Script ---- #

import os, re, time, pathlib, concurrent.futures, tempfile
from pathlib import Path
from urllib.parse import urlparse
import requests
import pandas as pd
import numpy as np


# --- selenium just to discover the PDF links reliably (page is JS-rendered) ---
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# --- table extraction ---
import tabula      # Java-backed; very good on these PDFs
import pdfplumber  # fallback if a table fails in tabula

FACTSHEETS_INDEX = "https://gco.iarc.who.int/today/en/fact-sheets-populations#countries"
OUT_DIR = pathlib.Path("globocan_factsheets")
PDF_DIR = OUT_DIR / "pdf"
PDF_DIR.mkdir(parents=True, exist_ok=True)

def list_population_factsheet_pdfs(timeout=30):
    """Return list of (country_text, href) for all population fact-sheet PDFs."""
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-gpu")
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=opts)
    try:
        driver.get(FACTSHEETS_INDEX)

        # Some sessions show a cookie banner; try to accept if present.
        try:
            WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Accept') or contains(., 'accept')]"))
            ).click()
        except Exception:
            pass

        # Wait for links to render
        WebDriverWait(driver, timeout).until(
            EC.presence_of_all_elements_located((By.TAG_NAME, "a"))
        )
        links = driver.find_elements(By.CSS_SELECTOR, "a[href$='.pdf']")
        pdfs = []
        for a in links:
            href = a.getAttribute("href") if hasattr(a, "getAttribute") else a.get_attribute("href")
            text = a.text.strip()
            if href and "/media/globocan/factsheets/populations/" in href and href.endswith(".pdf"):
                pdfs.append((text or pathlib.Path(urlparse(href).path).name, href))
        # Deduplicate
        seen = set(); out = []
        for t,h in pdfs:
            if h not in seen:
                seen.add(h); out.append((t,h))
        return out
    finally:
        driver.quit()

def safe_filename(url):
    return pathlib.Path(urlparse(url).path).name

def download_one(url, dest_dir=PDF_DIR, sleep_sec=0.2):
    fn = dest_dir / safe_filename(url)
    if fn.exists() and fn.stat().st_size > 0:
        return fn
    hdrs = {"User-Agent": "research/archival (contact: your-email@example.com)"}
    with requests.get(url, headers=hdrs, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(fn, "wb") as f:
            for chunk in r.iter_content(1<<15):
                if chunk: f.write(chunk)
    time.sleep(sleep_sec)    # be polite
    return fn

# ---------- Helpers ----------

def filename_to_meta(path):
    """
    Derive country name and numeric code from filename like:
    '840-united-states-of-america-fact-sheet.pdf'
    """
    name = pathlib.Path(path).name
    m = re.match(r"(\d+)-([a-z0-9\-]+)-fact-sheet\.pdf", name)
    if not m:
        return {"pop_code": None, "country": name.replace("-fact-sheet.pdf","")}
    pop_code = int(m.group(1))
    country = m.group(2).replace("-", " ").title()
    # Handle some known acronyms better
    country = country.replace(" Usa", " USA").replace(" Uk", " UK")
    return {"pop_code": pop_code, "country": country}


HEADER_TOKENS = (
    "new cases", "deaths", "mortality", "5-year", "5 year", "prevalence",
    "cum.risk", "cum risk", "cumulative risk", "(%)", "percent", "rank", "number", "prop", "per 100"
)

SITE_HINTS = {
    "breast","prostate","colorectum","cervix","cervix uteri","lung","liver","stomach","ovary",
    "bladder","kidney","thyroid","leukaemia","leukemia","nhl","non-hodgkin","pancreas","brain",
    "cns","oesophagus","esophagus","gallbladder","larynx","melanoma","nasopharynx","lip","oral",
    "corpus uteri","testis","penis","vagina","vulva","salivary","hypopharynx","kaposi"
}

def _norm(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s) if s is not None else "").strip()
    s = s.replace("\u00A0"," ")  # NBSP
    return s

def _norm_low(s: str) -> str:
    s = _norm(s).lower()
    # simplify some tokens
    s = s.replace("5 year", "5-year").replace("per 100 000","per 100000")
    s = s.replace("–","-").replace("—","-").replace("’","'")
    # drop stray years
    s = re.sub(r"\b(19|20)\d{2}\b", "", s)
    return re.sub(r"\s+", " ", s).strip()

def _combine_header_rows_general(df: pd.DataFrame, max_header_rows=4) -> tuple[pd.DataFrame, list[str]]:
    """Combine top 1–3(–4) rows that look like headers into a single header row."""
    if df.empty:
        return df, list(df.columns)

    # work with strings and keep empties
    tmp = df.copy().astype(str).replace("nan","")

    # decide how many top rows are header rows
    header_rows = 1
    for r in range(1, min(max_header_rows, len(tmp))):
        row_txt = " ".join(_norm_low(x) for x in tmp.iloc[r].tolist())
        if any(tok in row_txt for tok in HEADER_TOKENS) or "cancer" in row_txt or "site" in row_txt:
            header_rows = r + 1
        else:
            # stop at the first row that clearly looks like data (contains many site names or many pure numbers)
            break

    head = tmp.iloc[:header_rows].replace({"": None}).ffill(axis=0).fillna("")
    combined = []
    for col_idx in range(head.shape[1]):
        parts = [_norm(head.iloc[r, col_idx]) for r in range(head.shape[0])]
        parts = [p for p in parts if p]
        # de-duplicate while preserving order
        seen = set(); ordered = []
        for p in parts:
            pl = p.lower()
            if pl not in seen:
                seen.add(pl); ordered.append(p)
        combined.append(" ".join(ordered))

    body = tmp.iloc[header_rows:].reset_index(drop=True)
    body.columns = [_norm(c) for c in combined]
    return body, combined

def _guess_cancer_col(columns: list[str], df_preview: pd.DataFrame) -> int:
    # header clues first
    for i, c in enumerate(columns):
        cl = c.lower()
        if "cancer" in cl or ("site" in cl and "5-year" not in cl and "new cases" not in cl and "deaths" not in cl):
            return i
    # content clues
    sample = df_preview.iloc[:30].fillna("").astype(str)
    best_i, best_hits = 0, -1
    for i, c in enumerate(columns):
        vals = sample[c].tolist()
        hits = sum(any(tok in v.lower() for tok in SITE_HINTS) for v in vals)
        if hits > best_hits:
            best_hits, best_i = hits, i
    return best_i

def _map_metric_name(col: str) -> str | None:
    """Return canonical metric name for a messy header, or None to keep it as-is."""
    cl = _norm_low(col)

    if cl in {"cancer", "cancer site", "site", "cancer type"}:
        return "Cancer"

    def has(*tokens): return all(t in cl for t in tokens)
    def anyof(*tokens): return any(t in cl for t in tokens)

    # incidence / new cases
    if anyof("new cases","incidence"):
        if "cum" in cl: return "Incidence_CumRisk"
        if "rank" in cl: return "Incidence_Rank"
        if "%" in cl or "percent" in cl or "(%)" in cl: return "Incidence_Percent"
        if "number" in cl or "no." in cl or re.search(r"\bno\b", cl): return "Incidence_Number"
        if cl.strip() in {"new cases","incidence"}: return "Incidence_Number"

    # mortality / deaths
    if anyof("deaths","mortality"):
        if "cum" in cl: return "Mortality_CumRisk"
        if "rank" in cl: return "Mortality_Rank"
        if "%" in cl or "percent" in cl or "(%)" in cl: return "Mortality_Percent"
        if "number" in cl or "no." in cl or re.search(r"\bno\b", cl): return "Mortality_Number"
        if cl.strip() in {"deaths","mortality"}: return "Mortality_Number"

    # prevalence (5-year)
    if anyof("prevalence","5-year"):
        if "per 100000" in cl or "prop" in cl or "proportion" in cl: return "Prevalence_per100k"
        if "number" in cl or "no." in cl or re.search(r"\bno\b", cl): return "Prevalence_Number"
        return None

    return None

def _rename_to_standard(columns: list[str]) -> dict[str,str]:
    out = {}
    for c in columns:
        mapped = _map_metric_name(c)
        if mapped:
            out[c] = mapped
    return out

def _numericify(df: pd.DataFrame) -> pd.DataFrame:
    num_cols = [c for c in df.columns if c != "Cancer"]
    for c in num_cols:
        s = (
            df[c].astype(str)
                 .str.replace("\u00A0","", regex=False)    # NBSP
                 .str.replace(r"[,\s]", "", regex=True)    # thousand seps
                 .str.replace(r"[^\d\.\-]", "", regex=True)
        )
        # if multiple dots crept in, keep the first
        s = s.str.replace(r"(?<=\d)\.(?=.*\.)", "", regex=True)
        df[c] = pd.to_numeric(s, errors="coerce")
    return df

def _score_candidate(df: pd.DataFrame) -> float:
    """Higher is better. Prefer wide tables with expected tokens and a texty first column."""
    if df is None or df.empty:
        return -1
    cols = [c.lower() for c in df.columns]
    token_hits = sum(any(tok in c for tok in ["new cases","deaths","prevalence","cum","rank","(%)","percent"]) for c in cols)
    # guess cancer col and check textiness
    ci = _guess_cancer_col(list(df.columns), df)
    textiness = (df.iloc[:30, ci].astype(str).str.contains(r"[A-Za-z]", na=False)).mean()
    return df.shape[0] * (df.shape[1] + token_hits) * (1.0 + textiness)


def _standardize_table(df_in: pd.DataFrame) -> pd.DataFrame | None:
    if df_in is None or df_in.empty:
        return None

    # flatten multiindex if any
    if isinstance(df_in.columns, pd.MultiIndex):
        df_in.columns = [_norm(" ".join([str(x) for x in tup if str(x) != "None"])) for tup in df_in.columns]
    else:
        df_in.columns = [_norm(c) for c in df_in.columns]

    # combine top header rows (1–3)
    df, combined_cols = _combine_header_rows_general(df_in, max_header_rows=4)

    # identify cancer/site column; move it to first and call it "Cancer"
    cand_idx = _guess_cancer_col(list(df.columns), df)
    cols = list(df.columns)
    cols[0], cols[cand_idx] = cols[cand_idx], cols[0]
    df = df[cols].rename(columns={cols[0]: "Cancer"})

    # drop obvious header echoes / junk rows
    df["Cancer"] = df["Cancer"].apply(_norm)
    df = df[df["Cancer"].str.len() > 0]
    df = df[~df["Cancer"].str.contains(r"^(new cases|deaths|5[- ]?year|incidence|mortality|prevalence)$", flags=re.I, na=False)]

    # rename metrics
    rename_map = _rename_to_standard(df.columns[1:])
    df = df.rename(columns=rename_map)

    # keep known columns if present
    wanted = ["Cancer",
              "Incidence_Number","Incidence_Rank","Incidence_Percent","Incidence_CumRisk",
              "Mortality_Number","Mortality_Rank","Mortality_Percent","Mortality_CumRisk",
              "Prevalence_Number","Prevalence_per100k"]
    present = [c for c in wanted if c in df.columns]
    if "Cancer" not in present:
        return None
    df = df[present].copy()

    # sanity: should have multiple metric columns and ≥ ~10 rows
    if df.shape[0] < 10 or len([c for c in df.columns if c != "Cancer"]) < 3:
        return None

    df = _numericify(df)

    # usually good to drop “All cancers” totals (optional)
    df = df[~df["Cancer"].str.contains(r"^All cancers", flags=re.I, na=False)].reset_index(drop=True)

    return df
# ---------- Extraction core ----------

def parse_fact_sheet_table(pdf_path):
    """
    Parse the big 'Incidence, Mortality and Prevalence by cancer site' table on page 2.
    Returns the best recognized tidy DataFrame.
    """
    candidates = []

    # 1) Tabula lattice (let it guess)
    try:
        dfs_lat = tabula.read_pdf(
            str(pdf_path),
            pages="2",
            lattice=True,
            stream=False,
            guess=True,
            multiple_tables=True,
            pandas_options={"dtype": str}
        ) or []
        # keep plausibly-wide tables
        dfs_lat = [d for d in dfs_lat if isinstance(d, pd.DataFrame) and d.shape[1] >= 4]
        for d in dfs_lat:
            std = _standardize_table(d)
            if std is not None:
                candidates.append(std)
    except Exception:
        pass

    # 2) Tabula stream (whitespace)
    try:
        dfs_str = tabula.read_pdf(
            str(pdf_path),
            pages="2",
            lattice=False,
            stream=True,
            guess=True,
            multiple_tables=True,
            pandas_options={"dtype": str}
        ) or []
        dfs_str = [d for d in dfs_str if isinstance(d, pd.DataFrame) and d.shape[1] >= 4]
        for d in dfs_str:
            std = _standardize_table(d)
            if std is not None:
                candidates.append(std)
    except Exception:
        pass

    # pick the best candidate from Tabula if any
    if candidates:
        candidates.sort(key=_score_candidate, reverse=True)
        return candidates[0]

    # 2) Fallback: pdfplumber with lines strategy (robust to some PDFs where Tabula splits merged cells)
    with pdfplumber.open(str(pdf_path)) as pdf:
        page = pdf.pages[1]  # page 2 (0-indexed)
        table_settings = {
            "vertical_strategy": "lines",
            "horizontal_strategy": "lines",
            "snap_tolerance": 3,
            "join_tolerance": 2,
            "edge_min_length": 3,
            "min_words_vertical": 1,
            "min_words_horizontal": 1,
            "intersection_tolerance": 3,
            # use the *text_* options supported by new pdfplumber:
            "text_x_tolerance": 2,
            "text_y_tolerance": 2,
            "text_keep_blank_chars": True,
            }
        tables = page.extract_tables(table_settings=table_settings) or []
        # choose biggest table
        tables.sort(key=lambda t: (len(t) * len(t[0]) if t and t[0] else 0), reverse=True)
        for t in tables:
            if not t or not t[0]:
                continue
            df = pd.DataFrame(t[1:], columns=_clean_columns(t[0]))
            try:
                out = _standardize_table(df)
                if len(out) >= 10 and any(c != "Cancer" for c in out.columns):
                    return out
            except Exception:
                continue

    raise RuntimeError("Failed to parse the page-2 site-by-metrics table from this PDF.")


def main():
    """print("Discovering fact-sheet PDFs…")
    links = list_population_factsheet_pdfs()
    print(f"Found {len(links)} PDFs")

    # Download politely, a few at a time
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
        futures = [ex.submit(download_one, href) for _, href in links]
        pdf_paths = [f.result() for f in concurrent.futures.as_completed(futures)]
    pdf_paths.sort()"""
    PDF_DIR = Path("globocan_factsheets/pdf")
    OUT_DIR = Path("globocan_factsheets")
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    pdf_paths = sorted(PDF_DIR.glob("*.pdf"))
    if not pdf_paths:
        raise SystemExit(f"No PDFs found in {PDF_DIR.resolve()}")

    print("Parsing page-2 tables… (this may take a few minutes)")
    results = []
    failures = []

    for p in pdf_paths:
        meta = filename_to_meta(p)
        try:
            df = parse_fact_sheet_table(p)
            # annotate with metadata
            df.insert(0, "pop_code", meta["pop_code"])
            df.insert(1, "country", meta["country"])
            df.insert(2, "source_pdf", p.name)
            results.append(df)
            print(f"OK: {p.name} → {len(df)} rows")
        except Exception as e:
            err = f"{type(e).__name__}: {e}"
            failures.append({"pdf": p.name, "error": err})
            print(f"FAILED: {p.name} -> {err}")

    if not results:
        raise SystemExit("No tables parsed. Check Java/Tabula (Java) and pdfplumber and try again.")

    big = pd.concat(results, ignore_index=True, sort=False)

    # Normalize column order: metadata first, then the expected metrics if present
    wanted = [
        "Cancer",
        "Incidence_Number", "Incidence_Rank", "Incidence_Percent", "Incidence_CumRisk",
        "Mortality_Number", "Mortality_Rank", "Mortality_Percent", "Mortality_CumRisk",
        "Prevalence_Number", "Prevalence_per100k",
    ]
    ordered = ["pop_code", "country", "source_pdf"] + [c for c in wanted if c in big.columns]
    # keep any extra columns (if present) at the end
    ordered += [c for c in big.columns if c not in ordered]
    big = big[ordered]

    # Save outputs
    csv_path = OUT_DIR / "globocan2022_fact_sheets_page2.csv"
    pq_path  = OUT_DIR / "globocan2022_fact_sheets_page2.parquet"
    big.to_csv(csv_path, index=False)
    big.to_parquet(pq_path, index=False)

    # Save a failure log if any
    if failures:
        pd.DataFrame(failures).to_csv(OUT_DIR / "parse_failures.csv", index=False)

    print(f"Done → {csv_path} "
          f"({len(big)} rows from {len(results)} PDFs; {len(failures)} failures)")
    

if __name__ == "__main__":
    main()


selenium already installed.
webdriver-manager already installed.
requests already installed.
pandas already installed.
tabula-py already installed.
pdfplumber already installed.
pyarrow already installed.
Parsing page-2 tables… (this may take a few minutes)


/var/folders/4k/pcydymq12vx9n7hfl5zsqrf80000gn/T/ipykernel_91828/3006147339.py:292: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df["Cancer"].str.contains(r"^(new cases|deaths|5[- ]?year|incidence|mortality|prevalence)$", flags=re.I, na=False)]
/var/folders/4k/pcydymq12vx9n7hfl5zsqrf80000gn/T/ipykernel_91828/3006147339.py:292: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df["Cancer"].str.contains(r"^(new cases|deaths|5[- ]?year|incidence|mortality|prevalence)$", flags=re.I, na=False)]
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
/var/folders/4k/pcydymq12vx9n7hfl5zsqrf80000gn/T/ipykernel_91828/3006147339.py:292: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df

FAILED: 12-algeria-fact-sheet.pdf -> RuntimeError: Failed to parse the page-2 site-by-metrics table from this PDF.


/var/folders/4k/pcydymq12vx9n7hfl5zsqrf80000gn/T/ipykernel_91828/3006147339.py:292: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df["Cancer"].str.contains(r"^(new cases|deaths|5[- ]?year|incidence|mortality|prevalence)$", flags=re.I, na=False)]
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
/var/folders/4k/pcydymq12vx9n7hfl5zsqrf80000gn/T/ipykernel_91828/3006147339.py:292: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df["Cancer"].str.contains(r"^(new cases|deaths|5[- ]?year|incidence|mortality|prevalence)$", flags=re.I, na=False)]
/var/folders/4k/pcydymq12vx9n7hfl5zsqrf80000gn/T/ipykernel_91828/3006147339.py:292: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df

FAILED: 24-angola-fact-sheet.pdf -> RuntimeError: Failed to parse the page-2 site-by-metrics table from this PDF.


...

/var/folders/4k/pcydymq12vx9n7hfl5zsqrf80000gn/T/ipykernel_91828/3006147339.py:292: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df["Cancer"].str.contains(r"^(new cases|deaths|5[- ]?year|incidence|mortality|prevalence)$", flags=re.I, na=False)]
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats


FAILED: 8-albania-fact-sheet.pdf -> RuntimeError: Failed to parse the page-2 site-by-metrics table from this PDF.


/var/folders/4k/pcydymq12vx9n7hfl5zsqrf80000gn/T/ipykernel_91828/3006147339.py:292: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df["Cancer"].str.contains(r"^(new cases|deaths|5[- ]?year|incidence|mortality|prevalence)$", flags=re.I, na=False)]
/var/folders/4k/pcydymq12vx9n7hfl5zsqrf80000gn/T/ipykernel_91828/3006147339.py:292: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df = df[~df["Cancer"].str.contains(r"^(new cases|deaths|5[- ]?year|incidence|mortality|prevalence)$", flags=re.I, na=False)]


SystemExit: No tables parsed. Check Java/Tabula (Java) and pdfplumber and try again.

/Users/archie/Documents/Visual Studio Code/Geospatial-modelling-radiotherapy-access/Geospatial-modelling-radiotherapy-access/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
